# Delta Lake com PySpark
## Cenário

Loja virtual fictícia com duas tabelas: **clientes** e **pedidos** (1 cliente : N pedidos).
O conjunto de dados foi **criado pelo grupo** (`data/clientes.csv` e `data/pedidos.csv`).

![Modelo ER](../../assets/modelo_er.svg)

In [1]:
import sys; sys.path.append("../..")
from pesquisadados.main import spark_delta as criar_spark, DADOS
spark = criar_spark()
spark.version

26/10/03 21:29:30 WARN Utils: Your hostname, DESKTOP-73OKR5D resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/03 21:29:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/thomasmaggi/pesquisadados/pesquisadados/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thomasmaggi/.ivy2/cache
The jars for the packages stored in: /home/thomasmaggi/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-87daa293-aa24-434e-9290-faa470885535;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 338ms :: artifacts dl 12ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |  

'3.5.3'

## DDL

In [2]:
import shutil
shutil.rmtree(DADOS / "delta", ignore_errors=True)
spark.sql("DROP TABLE IF EXISTS clientes")
spark.sql("DROP TABLE IF EXISTS pedidos")
spark.sql(f"""
CREATE TABLE clientes (
  id_cliente INT, nome STRING, cidade STRING, uf STRING
) USING delta LOCATION '{DADOS}/delta/clientes'
""")
spark.sql(f"""
CREATE TABLE pedidos (
  id_pedido INT, id_cliente INT, produto STRING, quantidade INT,
  valor_unitario DECIMAL(10,2), data_pedido DATE, status STRING
) USING delta LOCATION '{DADOS}/delta/pedidos'
""")

DataFrame[]

## Carga inicial (INSERT a partir do CSV)

In [3]:
clientes = spark.read.csv(str(DADOS/"clientes.csv"), header=True, inferSchema=True)
pedidos = spark.read.csv(str(DADOS/"pedidos.csv"), header=True, inferSchema=True)
clientes.createOrReplaceTempView("v_clientes"); pedidos.createOrReplaceTempView("v_pedidos")
spark.sql("INSERT INTO clientes SELECT id_cliente, nome, cidade, uf FROM v_clientes")
spark.sql("""INSERT INTO pedidos
SELECT id_pedido, id_cliente, produto, quantidade, CAST(valor_unitario AS DECIMAL(10,2)),
       CAST(data_pedido AS DATE), status FROM v_pedidos""")
spark.table("pedidos").show()

26/10/03 21:29:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---------+----------+--------+----------+--------------+-----------+---------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|   status|
+---------+----------+--------+----------+--------------+-----------+---------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|     PAGO|
|      102|         2|   Mouse|         2|         80.00| 2026-09-02|     PAGO|
|      103|         3| Monitor|         1|       1200.00| 2026-09-03| PENDENTE|
|      104|         1| Teclado|         1|        250.00| 2026-09-05| PENDENTE|
|      105|         4| Headset|         1|        300.00| 2026-09-06|CANCELADO|
+---------+----------+--------+----------+--------------+-----------+---------+



## INSERT
Insere um novo pedido.

In [4]:
spark.sql("""INSERT INTO pedidos VALUES
(106, 2, 'Webcam', 1, 350.00, DATE'2026-09-10', 'PENDENTE')""")
spark.table("pedidos").orderBy("id_pedido").show()

+---------+----------+--------+----------+--------------+-----------+---------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|   status|
+---------+----------+--------+----------+--------------+-----------+---------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|     PAGO|
|      102|         2|   Mouse|         2|         80.00| 2026-09-02|     PAGO|
|      103|         3| Monitor|         1|       1200.00| 2026-09-03| PENDENTE|
|      104|         1| Teclado|         1|        250.00| 2026-09-05| PENDENTE|
|      105|         4| Headset|         1|        300.00| 2026-09-06|CANCELADO|
|      106|         2|  Webcam|         1|        350.00| 2026-09-10| PENDENTE|
+---------+----------+--------+----------+--------------+-----------+---------+



## UPDATE
Marca como PAGO os pedidos pendentes do cliente 1.

In [5]:
spark.sql("UPDATE pedidos SET status = 'PAGO' WHERE id_cliente = 1 AND status = 'PENDENTE'")
spark.table("pedidos").orderBy("id_pedido").show()

+---------+----------+--------+----------+--------------+-----------+---------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|   status|
+---------+----------+--------+----------+--------------+-----------+---------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|     PAGO|
|      102|         2|   Mouse|         2|         80.00| 2026-09-02|     PAGO|
|      103|         3| Monitor|         1|       1200.00| 2026-09-03| PENDENTE|
|      104|         1| Teclado|         1|        250.00| 2026-09-05|     PAGO|
|      105|         4| Headset|         1|        300.00| 2026-09-06|CANCELADO|
|      106|         2|  Webcam|         1|        350.00| 2026-09-10| PENDENTE|
+---------+----------+--------+----------+--------------+-----------+---------+



## DELETE
Remove os pedidos cancelados.

In [6]:
spark.sql("DELETE FROM pedidos WHERE status = 'CANCELADO'")
spark.table("pedidos").orderBy("id_pedido").show()

+---------+----------+--------+----------+--------------+-----------+--------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|  status|
+---------+----------+--------+----------+--------------+-----------+--------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|    PAGO|
|      102|         2|   Mouse|         2|         80.00| 2026-09-02|    PAGO|
|      103|         3| Monitor|         1|       1200.00| 2026-09-03|PENDENTE|
|      104|         1| Teclado|         1|        250.00| 2026-09-05|    PAGO|
|      106|         2|  Webcam|         1|        350.00| 2026-09-10|PENDENTE|
+---------+----------+--------+----------+--------------+-----------+--------+



## Histórico de versões

In [7]:
spark.sql("DESCRIBE HISTORY pedidos").select("version","operation").show()

+-------+------------+
|version|   operation|
+-------+------------+
|      4|      DELETE|
|      3|      UPDATE|
|      2|       WRITE|
|      1|       WRITE|
|      0|CREATE TABLE|
+-------+------------+



In [8]:
spark.stop()